In [21]:
import pandas as pd

In [22]:
data = pd.read_csv('task_2_data_ex.csv')

In [23]:
db = data[['plant_id', 
            'produced_material', 
            'produced_material_release_type', 
            'produced_material_production_type',
            'produced_material_quantity',
            'component_material',
            'component_material_release_type',
            'component_material_production_type',
            'component_material_quantity',
            'year']].copy()

fin = db[db['produced_material_release_type']=='FIN'].rename(columns={
            'plant_id': 'plant',
            'produced_material': 'fin_material_id',
            'produced_material_release_type': 'fin_material_release_type',
            'produced_material_production_type': 'fin_material_production_type',
            'produced_material_quantity': 'fin_production_quantity',
            'component_material': 'prod_material_id',
            'component_material_release_type': 'prod_material_release_type',
            'component_material_production_type': 'prod_material_production_type',
            'component_material_quantity': 'prod_material_production_quantity'
            })

prod = db[db['produced_material_release_type']=='PROD']
prod = prod.rename(columns={'component_material': 'merge_component_material'})
result = []

In [24]:
def unpack_step(step_df, prod_df, columns_to_merge):
    merged = step_df.merge(prod_df[columns_to_merge], 
                           left_on=['plant', 'year', 'component_material'], 
                           right_on=['plant_id', 'year', 'produced_material'], 
                           how='inner')
    
    merged = merged.drop(columns=['component_material', 'plant_id'])
    merged = merged.rename(columns={'merge_component_material': 'component_material'})

    is_intermediate = merged['component_material_release_type'] == 'PROD'

    step_finished = merged[~is_intermediate].copy().drop(columns=['produced_material'])

    step_in_process = merged[is_intermediate].copy().drop(columns=[
        'produced_material',
        'component_material_release_type',
        'component_material_production_type', 
        'component_material_quantity'
    ])
    
    return step_finished, step_in_process

In [25]:
columns = ['produced_material', 
           'merge_component_material', 
           'component_material_release_type',
           'component_material_production_type', 
           'component_material_quantity',
           'plant_id',
           'year']

in_process = fin.copy()
in_process['component_material'] = in_process['prod_material_id']
result = pd.DataFrame()

In [26]:
while not in_process.empty:
    finished_batch, in_process = unpack_step(in_process, prod, columns)
    result = pd.concat([result, finished_batch], ignore_index=True)

In [27]:
result = result.rename(columns={
    'component_material': 'component_id',
    'component_material_release_type': 'component_material_release_type',
    'component_material_production_type': 'component_material_production_type',
    'component_material_quantity': 'component_consumption_quantity'
})
result = result.sort_values(by='year', ascending=True, ignore_index=True)
result

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,year,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity
0,RLT_10,10000,FIN,8002,990.00,50000,PROD,8002.0,990.00,2024,90000,ADD,NaN,50.00
1,RLT_10,10000,FIN,8002,990.00,50000,PROD,8002.0,990.00,2024,90001,ADD,NaN,20.00
2,RLT_10,10000,FIN,8002,990.00,50000,PROD,8002.0,990.00,2024,90000,ADD,NaN,49.00
3,RLT_10,10000,FIN,8002,990.00,50000,PROD,8002.0,990.00,2024,90001,ADD,NaN,22.00
4,RLT_10,10000,FIN,8002,990.00,50000,PROD,8002.0,990.00,2024,90000,ADD,NaN,46.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5221435,RLT_14,10009,FIN,8002,"1,081.00",50009,PROD,8002.0,"1,081.00",2024,90050,ADD,NaN,67.00
5221436,RLT_14,10009,FIN,8002,"1,081.00",50009,PROD,8002.0,"1,081.00",2024,70009,RM,NaN,"2,558.00"
5221437,RLT_14,10009,FIN,8002,"1,081.00",50009,PROD,8002.0,"1,081.00",2024,90050,ADD,NaN,69.00
5221438,RLT_14,10009,FIN,8002,"1,081.00",50009,PROD,8002.0,"1,081.00",2024,70009,RM,NaN,"2,523.00"
